# 01 — Gamma-or-Skip Policy Preview

Apply this version's offline policy to a fixed random sample of calibration originals to inspect routing, per-image cost and before/after pixels. Nothing is exported, trained or tuned here.

**Inputs:** v1.yaml (this folder), the shared split table and original calibration images.

**Outputs:** In-memory tables and figures only.

**Prerequisites:** Original BDD sources under the shared data_root. No upstream notebook is executed.

**Outline**

- [1. Configuration](#section-1)
- [2. Policy on Calibration Samples](#section-2)
- [3. Policy Summary](#section-3)

<a id="section-1"></a>

## 1. Configuration

Change SAMPLES/SHOW freely; policy parameters live only in the version YAML.

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from vehicle_detection.common import read_image, require_paths
from vehicle_detection.experiment_data import load_experiment, shared_splits
from vehicle_detection.preprocess import apply_recipe

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
EXPERIMENT = "v1"
require_paths(
    config=ROOT / f"notebooks/experiments/{EXPERIMENT}/{EXPERIMENT}.yaml",
    shared=ROOT / "configs/data.yaml",
)
config = load_experiment(ROOT, EXPERIMENT)

SAMPLES = 64
SHOW = 4
shared, rows = shared_splits(ROOT)
data_root = ROOT / shared["data_root"]
require_paths(data_root=data_root)

<a id="section-2"></a>

## 2. Policy on Calibration Samples

Features are measured on the original pixels; only the routed operator changes the image.

In [ ]:
calibration = [r for r in rows if r["role"] == "calibration"]
picked = np.random.default_rng(config["seed"]).choice(len(calibration), SAMPLES, replace=False)
records, pairs = [], []
for index in picked:
    row = calibration[index]
    image = read_image(data_root / row["source_path"])
    start = time.perf_counter()
    output, applied, scores = apply_recipe(image, config["variant"], config["recipes"])
    elapsed = (time.perf_counter() - start) * 1000
    records.append(
        {"id": row["id"], "applied": "+".join(a["name"] for a in applied), "ms": elapsed, **scores}
    )
    if applied and len(pairs) < SHOW:
        pairs.append((row["id"], image, output))
preview = pd.DataFrame(records)
display(
    preview.groupby("applied", dropna=False).agg(
        images=("id", "size"),
        ms=("ms", "mean"),
        brightness=("brightness", "mean"),
        contrast=("contrast", "mean"),
    )
)
for sample_id, before, after in pairs:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), layout="constrained")
    for ax, image, label in zip(axes, (before, after), ("Original", config["name"]), strict=True):
        ax.imshow(image[:, :, ::-1])
        ax.set(title=f"{sample_id} · {label}", xticks=[], yticks=[])
    plt.show()

<a id="section-3"></a>

## 3. Policy Summary

In [ ]:
print("Variant:", config["variant"], "| Recipes:", config["recipes"])
print(f"Applied to {(preview.applied != '').mean():.1%} of {len(preview)} calibration samples")
print("Export happens only in 02_train_prepare; final-test pixels stay sealed.")